# American and Exotic Option Methods

Module: Derivatives

## Lesson summary

Many options cannot be handled by a single closed-form Black-Scholes price. American exercise, path dependence, averaging, and barriers require numerical methods. This lesson compares CRR convergence, Leisen-Reimer American put pricing, Asian option control variates, and the Broadie-Glasserman-Kou continuity correction for discretely monitored barriers.

## Learning objectives

By the end of this lesson, students should be able to:

- compare CRR binomial convergence against a Black-Scholes benchmark;
- explain early exercise in American put valuation;
- use a control variate to reduce Monte Carlo error for an Asian call;
- apply the BGK continuity correction to a discrete barrier;
- state the model-risk tradeoffs of lattice and simulation methods.

## Python setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.derivatives import (
    arithmetic_asian_call_control_variate,
    bgk_adjusted_barrier,
    black_scholes_price,
    crr_binomial_option_price,
    leisen_reimer_american_put,
)

## CRR convergence

The CRR tree converges to Black-Scholes for European options, but the convergence can oscillate because terminal nodes do not always align well with the payoff kink.

In [ ]:
spot = 100.0
strike = 100.0
rate = 0.05
volatility = 0.25
maturity = 1.0
dividend_yield = 0.0

benchmark = black_scholes_price(spot, strike, rate, volatility, maturity, "call", dividend_yield)
steps_grid = [5, 10, 25, 50, 100, 250, 500]

convergence = pd.DataFrame({"steps": steps_grid})
convergence["crr_call"] = [
    crr_binomial_option_price(
        spot,
        strike,
        rate,
        volatility,
        maturity,
        steps=steps,
        option_type="call",
        dividend_yield=dividend_yield,
    )
    for steps in steps_grid
]
convergence["black_scholes_call"] = benchmark
convergence["error"] = convergence["crr_call"] - benchmark
convergence

In [ ]:
ax = convergence.plot(x="steps", y="error", marker="o", figsize=(8, 4), legend=False)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("CRR Convergence Error")
ax.set_xlabel("Steps")
ax.set_ylabel("Price error")
ax.grid(True, alpha=0.3)
plt.show()

## American put early exercise

American options require an optimal stopping check at every node:

$$
V = \max(\text{continuation value}, \text{intrinsic value}).
$$

In [ ]:
american_comparison = pd.DataFrame(
    {
        "method": ["European Black-Scholes", "CRR American", "Leisen-Reimer American"],
        "put_price": [
            black_scholes_price(spot, strike, rate, volatility, maturity, "put"),
            crr_binomial_option_price(
                spot,
                strike,
                rate,
                volatility,
                maturity,
                steps=501,
                option_type="put",
                american=True,
            ),
            leisen_reimer_american_put(
                spot,
                strike,
                rate,
                volatility,
                maturity,
                steps=501,
            ),
        ],
    }
)
american_comparison

## Asian option control variate

Arithmetic Asian options usually need simulation. A geometric Asian option is a useful control variate because it is highly correlated with the arithmetic payoff and has a known analytical price.

In [ ]:
asian = arithmetic_asian_call_control_variate(
    spot=100,
    strike=100,
    rate=0.05,
    volatility=0.25,
    maturity=1.0,
    observations=52,
    paths=20_000,
    seed=2027,
)

pd.Series(asian.__dict__)

In [ ]:
asian.naive_standard_error / asian.control_variate_standard_error

The ratio above is the standard-error reduction factor. Even when both estimators are unbiased for the same target only approximately in finite samples, the control variate typically gives much tighter estimates.

## Barrier continuity correction

For discretely monitored barriers, the BGK adjustment shifts the barrier away from the spot to approximate the difference between continuous and discrete monitoring:

$$
H_{adj} = H \exp(\pm \beta \sigma \sqrt{\Delta t}),
\qquad
\beta \approx 0.5826.
$$

In [ ]:
barrier = 80.0
daily_dt = 1 / 252

pd.Series(
    {
        "physical_down_barrier": barrier,
        "bgk_adjusted_down_barrier": bgk_adjusted_barrier(
            barrier,
            volatility=0.25,
            monitoring_interval=daily_dt,
            barrier_type="down",
        ),
        "physical_up_barrier": 120.0,
        "bgk_adjusted_up_barrier": bgk_adjusted_barrier(
            120.0,
            volatility=0.25,
            monitoring_interval=daily_dt,
            barrier_type="up",
        ),
    }
)

## Model limitations

- American and exotic option prices are sensitive to numerical method, monitoring convention, and exercise policy.
- Control variates reduce simulation noise only when the control remains highly correlated with the target payoff.
- Barrier corrections are approximations and can fail when barriers are close to spot or volatility is unstable.